# PRAGMA · A‑E1 etapa 1 — SAM 2 automático (4 configuraciones) · un clic

> **No tienes que decidir ni mirar nada.** Arrastra la foto `P1070614.JPG` al panel **Archivos**
> (icono de carpeta, a la izquierda) y pulsa **Entorno de ejecución → Ejecutar todas**. Al final se
> descarga un ZIP: **adjúntalo solo a Claude**.

Qué hace, sin intervención:

0. Comprueba que la GPU sirve (L4 o mejor, con bfloat16). **Si no, se detiene al instante** y te dice
   qué cambiar: una corrida en otra GPU no contaría.
1. Instala SAM 2 **en el commit exacto** `2b90b9f5` y descarga SAM 2.1 Large. **Se detiene** si el
   commit o el checkpoint no son los congelados.
2. Encuentra la foto por su huella SHA‑256.
3. Ejecuta las **4 configuraciones** del generador automático de máscaras prerregistradas
   (AMG‑0 a AMG‑3), en orden, con los parámetros exactos del protocolo.
4. Empaqueta todas las máscaras con un manifiesto de hashes en
   `PRAGMA_AE1S1_<run>_PENDING_ANALYSIS.zip`.

**Por qué no enseña resultados:** el análisis es externo y está fijado antes de la corrida. Parte de
él es una revisión ciega a doble llave. No compartas capturas de este cuaderno con ChatGPT ni con nadie.

Protocolo: `ae1/AE1_STAGE1_READING_PROTOCOL.json` (SHA‑256 del archivo `9c459ee3aac4b1efc94509c58ab81856bb9d49561b2b91ca7100785e2934aead`;
`content_sha256` `cee53ee86fa158dbf7398ae5c19d35cadc5ce5008920f671120502c5b45e271d`). Techo de esta etapa: `INCONCLUSIVE_GT_INCOMPLETE`, nunca un PASS.

## 0. Antes de pulsar «Ejecutar todas»

1. **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU → L4.** Usa **L4**, como en
   las corridas anteriores. Si eliges otra GPU sin bfloat16 (por ejemplo, T4) o ninguna, la primera celda
   se detiene con `FAIL_ENVIRONMENT`: cambia a L4 y vuelve a pulsar **Ejecutar todas**.
2. Arrastra `P1070614.JPG` al panel **Archivos**. Si no lo haces, la celda 2 te pedirá la foto con
   un botón **Elegir archivos**. El nombre no importa: se reconoce por su huella.
3. **Entorno de ejecución → Ejecutar todas.** Tarda unos minutos: instalar SAM 2, descargar 857 MiB y
   generar las máscaras de las 4 configuraciones. Al terminar, el navegador descarga el ZIP.

Si aparece un error en rojo, copia el texto y pégalo a Claude. No cambies nada del cuaderno.

In [ ]:
# Celda 0 · compuerta de GPU (ChatGPT 020): solo cuenta CUDA con capacidad ≥ 8, que tiene bfloat16 nativo
import torch
if not torch.cuda.is_available():
    raise RuntimeError("FAIL_ENVIRONMENT: no hay GPU. Entorno de ejecución → Cambiar tipo de entorno de "
                       "ejecución → L4 → Guardar, y vuelve a pulsar «Ejecutar todas».")
_capability = torch.cuda.get_device_capability(0)
if _capability[0] < 8:
    raise RuntimeError(f"FAIL_ENVIRONMENT: {torch.cuda.get_device_name(0)} (capacidad {_capability[0]}.{_capability[1]}) "
                       "no tiene bfloat16 nativo y la corrida no contaría. Cambia el entorno a L4 y vuelve a "
                       "pulsar «Ejecutar todas».")
print(f"GPU válida: {torch.cuda.get_device_name(0)} · capacidad {_capability[0]}.{_capability[1]} · bfloat16")

In [ ]:
# Celda 1 · SAM 2 en el commit congelado y checkpoint verificado (red; en el arnés local se omite)
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

SAM2_PINNED_COMMIT = "2b90b9f5ceec907a1c18123530e92e794ad901a4"
CHECKPOINT_SHA256_PINNED = "2647878d5dfa5098f2f8649825738a9345572bae2d4350a2468587ece47dd318"
CHECKPOINT_BYTES_PINNED = 898083611
REPO_DIR = Path("/content/pragma_sam2_2b90b9f5")
WORK_DIR = Path("/content/pragma_run")
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
CHECKPOINT = CHECKPOINT_DIR / "sam2.1_hiera_large.pt"
CHECKPOINT_URL = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt"
MODEL_CFG = "configs/sam2.1/sam2.1_hiera_l.yaml"
WORK_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def run_checked(args, **kwargs):
    print("$", " ".join(map(str, args)))
    return subprocess.run(args, check=True, text=True, **kwargs)

if not (REPO_DIR / ".git").exists():
    # Clon sin blobs y checkout del commit exacto (no «main», que puede moverse).
    run_checked(["git", "clone", "--filter=blob:none", "--no-checkout",
                 "https://github.com/facebookresearch/sam2.git", str(REPO_DIR)])
run_checked(["git", "-C", str(REPO_DIR), "checkout", "--quiet", SAM2_PINNED_COMMIT])
SAM2_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if SAM2_COMMIT != SAM2_PINNED_COMMIT:
    raise RuntimeError(f"FAIL_FREEZE: SAM 2 está en {SAM2_COMMIT}, no en {SAM2_PINNED_COMMIT}. No se genera nada.")

install_env = os.environ.copy()
install_env["SAM2_BUILD_CUDA"] = "0"
run_checked([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=REPO_DIR, env=install_env)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
import importlib
importlib.invalidate_caches()
import sam2
print("SAM 2 importable:", Path(sam2.__file__).resolve())

def download(url, destination):
    temporary = destination.with_suffix(destination.suffix + ".part")
    def progress(blocks, block_size, total):
        if total > 0 and blocks % 128 == 0:
            print(f"Descarga: {min(100, blocks*block_size*100/total):5.1f}%", end="\r")
    urllib.request.urlretrieve(url, temporary, reporthook=progress)
    temporary.replace(destination)
    print(f"\nCheckpoint: {destination.stat().st_size / 2**20:.1f} MiB")

if not CHECKPOINT.exists() or CHECKPOINT.stat().st_size != CHECKPOINT_BYTES_PINNED:
    download(CHECKPOINT_URL, CHECKPOINT)
digest = hashlib.sha256()
with open(CHECKPOINT, "rb") as handle:
    for block in iter(lambda: handle.read(1 << 20), b""):
        digest.update(block)
if CHECKPOINT.stat().st_size != CHECKPOINT_BYTES_PINNED or digest.hexdigest() != CHECKPOINT_SHA256_PINNED:
    raise RuntimeError("FAIL_FREEZE: el checkpoint no es el congelado (bytes o SHA-256). No se genera nada.")
os.chdir(WORK_DIR)
print("Congelado verificado · SAM 2", SAM2_COMMIT, "· checkpoint", CHECKPOINT_SHA256_PINNED[:12], "…")

In [ ]:
# Celda 2 · hardware, precisión y foto (se reconoce por SHA-256; el nombre puede variar)
EXPECTED_IMAGE_SHA256 = "8f6e3b6f5013265a45c7e89121e3f0a18e3386951e2e75378b28da6d02ec529d"
EXPECTED_IMAGE_BYTES = 4260352
import hashlib, io, json, os, platform, time, uuid, zipfile
from contextlib import nullcontext
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps
import torch
from google.colab import files

PRAGMA_HEADLESS = os.environ.get("PRAGMA_HEADLESS") == "1"   # 1 = Colab CLI, sin navegador

def select_precision(cuda_available, capability=None):
    if not cuda_available:
        return {"device": "cpu", "dtype": "float32", "autocast": False}
    native_bf16 = int(capability[0]) >= 8
    return {"device": "cuda", "dtype": "bfloat16" if native_bf16 else "float16", "autocast": True}

CUDA_AVAILABLE = torch.cuda.is_available()
CUDA_CAPABILITY = torch.cuda.get_device_capability(0) if CUDA_AVAILABLE else None
PRECISION = select_precision(CUDA_AVAILABLE, CUDA_CAPABILITY)
DEVICE = PRECISION["device"]
TORCH_DTYPE = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}[PRECISION["dtype"]]

def inference_precision():
    return torch.autocast(device_type="cuda", dtype=TORCH_DTYPE) if DEVICE == "cuda" else nullcontext()

def synchronize():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()
    DEVICE_NAME = torch.cuda.get_device_name(0)
else:
    DEVICE_NAME = "CPU"
    print("ADVERTENCIA: sin GPU. La corrida se registrará como REAL_CPU y no será comparable.")

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = WORK_DIR / "runs_ae1s1" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)

def sha256_file(path, chunk=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()

def find_acceptance_image(folders):
    for folder in map(Path, folders):
        if not folder.is_dir():
            continue
        for path in sorted(folder.iterdir()):
            if (path.is_file() and path.suffix.lower() in (".jpg", ".jpeg")
                    and path.stat().st_size == EXPECTED_IMAGE_BYTES and sha256_file(path) == EXPECTED_IMAGE_SHA256):
                return path
    return None

IMAGE_PATH = find_acceptance_image([Path("/content"), Path("/mnt/data"), WORK_DIR])
if IMAGE_PATH is not None:
    print(f"Foto encontrada por su huella SHA-256: {IMAGE_PATH}")
elif PRAGMA_HEADLESS:
    raise FileNotFoundError("Modo sin navegador: sube antes la foto a /content.")
else:
    print("Selecciona la foto P1070614.JPG. Se validará por SHA-256; el nombre puede variar.")
    uploaded = files.upload()
    exact = [(n, p) for n, p in uploaded.items() if hashlib.sha256(p).hexdigest() == EXPECTED_IMAGE_SHA256]
    if len(exact) != 1:
        raise ValueError("No se recibió exactamente la fotografía de aceptación.")
    IMAGE_PATH = RUN_DIR / "P1070614.JPG"
    IMAGE_PATH.write_bytes(exact[0][1])

IMAGE_SHA256 = sha256_file(IMAGE_PATH)
assert IMAGE_SHA256 == EXPECTED_IMAGE_SHA256, "La foto no coincide con la aceptación acordada."
image = np.asarray(ImageOps.exif_transpose(Image.open(IMAGE_PATH)).convert("RGB"))
assert image.shape[:2] == (2248, 4000), image.shape
ENVIRONMENT = {
    "python": platform.python_version(), "torch": torch.__version__, "device": DEVICE, "device_name": DEVICE_NAME,
    "cuda_capability": CUDA_CAPABILITY, "dtype": PRECISION["dtype"], "sam2_commit": SAM2_COMMIT,
    "checkpoint_bytes": CHECKPOINT.stat().st_size, "checkpoint_sha256": sha256_file(CHECKPOINT),
    "image_sha256": IMAGE_SHA256, "image_size": [image.shape[1], image.shape[0]], "run_id": RUN_ID,
}
print(json.dumps(ENVIRONMENT, indent=2, ensure_ascii=False))

In [ ]:
# Celda 3 · protocolo de lectura embebido (no editar): plan de las 4 llamadas y congelados
PROTOCOL_FILE_SHA256 = "9c459ee3aac4b1efc94509c58ab81856bb9d49561b2b91ca7100785e2934aead"
PROTOCOL_TEXT = "{\n  \"schema\": \"pragma.ae1_stage1_reading_protocol\",\n  \"schema_version\": \"0.2.0\",\n  \"status\": \"PREREGISTERED\",\n  \"gpu_gate\": \"HOLD: la GPU espera el GO de ChatGPT tras revisar esta versión, el cuaderno y su verificador (ChatGPT 020)\",\n  \"origin\": \"carta 019 (ORDEN 190) con las correcciones de ChatGPT 019 (ORDEN 191: R1 ciega por configuración, R2 con cotas a nivel de conjunto, R3 diagnóstico, R4 con una persona basta) y de ChatGPT 020 (ORDEN 201: triaje → revisión exhaustiva, compuerta bfloat16)\",\n  \"supersedes\": {\n    \"schema_version\": \"0.1.0\",\n    \"file_sha256\": \"810818759e402c4df870de80ea03211c1737935722c96e51d516a1cde535dc98\",\n    \"content_sha256\": \"d40cc5c4a3f3256421a40b28611611b68023bf515ae5e52bb2f6d836e928b57b\",\n    \"reviewed_in\": \"dialogo/020_chatgpt_a_claude.md (ORDEN 201): R1_TOP3_AS_EARLY_STOP = REJECTED, GPU_BF16_GATE = REQUIRED\"\n  },\n  \"binds\": {\n    \"contract\": {\n      \"file\": \"ae1/CONTRATO_ANALISIS_A-E1.json\",\n      \"sha256\": \"39180407b138f0f96b4a0e29f47403e909e4e44c9cc97c79b43c718bf24c874f\",\n      \"content_sha256\": \"96de8a3171653bfb6f0a81f3855bbec392aac80f136d6180e409c2f55750edfe\"\n    },\n    \"sweep\": {\n      \"file\": \"ae1/SWEEP_PRERREGISTRO_A-E1.json\",\n      \"sha256\": \"25a61aa98ec7db261d837ba4129474905f41bfedb28cc76e599ef920172d5a38\"\n    },\n    \"a_e0\": {\n      \"file\": \"ae0/scene_inventory.json\",\n      \"sha256\": \"765aaa4ae296c27e1e0924f511f268627ec834a80497e440574be7ffd1fa1d37\",\n      \"content_sha256\": \"0dba6767b0c7ca6d02a2f2498d7fb755f4eeeba65c88f7b3c798836f7136bb6b\",\n      \"reference_type\": \"AI_CONSENSUS_REFERENCE\"\n    },\n    \"reader\": {\n      \"file\": \"pragma_ae/ae1_stage1.py\",\n      \"sha256\": \"f31afde2bc502a55c9e331bc392b7e7ea8c2e2c2106af38790392403bf53ead6\"\n    },\n    \"metrics_implementation_sha256\": {\n      \"pragma_ae/metrics.py\": \"48b3ff9af53cd320adcb89eca82e95be4acb18272c5bfa958247e0075fb95e96\",\n      \"pragma_ae/masks.py\": \"412d1ead02fb92891ca989c5c6ac7a5b2dcd5b6bcdbb57551ec1ce12db712ac9\",\n      \"pragma_ae/inventory.py\": \"6540c0dcc21c718ebc3aee4b6a94d2680af536fd2680629f009ff197f2e42035\",\n      \"pragma_ae/keydiff.py\": \"c5e15ed6f1ab81b8b0237df9b1e5bd11849a97219c5e5af321caf634546d1e7a\"\n    }\n  },\n  \"stage\": {\n    \"name\": \"ETAPA_1\",\n    \"tier_a\": 38,\n    \"tier_a_with_mask\": [\n      \"ae0_001\",\n      \"ae0_002\",\n      \"ae0_003\"\n    ],\n    \"ceiling\": \"INCONCLUSIVE_GT_INCOMPLETE\",\n    \"never\": \"PASS_PROPOSALS: ningún informe de esta etapa la convierte en PASS, aunque los resultados sean buenos\",\n    \"role\": \"cribado de falsación / parada temprana, no validación completa\"\n  },\n  \"freeze\": {\n    \"SAM2_GIT_COMMIT\": \"2b90b9f5ceec907a1c18123530e92e794ad901a4\",\n    \"MODEL_CONFIG\": \"configs/sam2.1/sam2.1_hiera_l.yaml\",\n    \"CHECKPOINT_FILENAME\": \"sam2.1_hiera_large.pt\",\n    \"CHECKPOINT_SHA256\": \"2647878d5dfa5098f2f8649825738a9345572bae2d4350a2468587ece47dd318\",\n    \"IMAGE_SHA256\": \"8f6e3b6f5013265a45c7e89121e3f0a18e3386951e2e75378b28da6d02ec529d\",\n    \"IMAGE_H_x_W\": [\n      2248,\n      4000\n    ]\n  },\n  \"model\": {\n    \"build\": \"build_sam2(MODEL_CFG, CHECKPOINT, device=DEVICE, apply_postprocessing=False)\",\n    \"generator\": \"SAM2AutomaticMaskGenerator(model, **generator_kwargs).generate(image)\",\n    \"why_no_postprocessing\": \"es lo que hace el ejemplo oficial de AMG en el commit congelado\",\n    \"sources_at_commit\": {\n      \"notebooks/automatic_mask_generator_example.ipynb\": \"a503375b91ea76d463af8b12955e60270abede74937a2eda613eb3330ad56501\",\n      \"sam2/automatic_mask_generator.py\": \"66df266dbe14412305ae3398f0ec1bb21b303a93216b102d767e6c4ee5d4c3d7\",\n      \"sam2/build_sam.py\": \"856d64d71e44407401297b551884fb4cd1aba8082de0ff5fc60fac3dd42f094d\"\n    },\n    \"precision\": \"torch.autocast CUDA en bfloat16; exige capacidad CUDA ≥ 8 (L4, A100, H100). La celda 0 se detiene antes de instalar si no la hay (FAIL_ENVIRONMENT) y la celda 4 lo vuelve a exigir antes de cargar el modelo\",\n    \"image\": \"RGB HWC uint8 tras exif_transpose, 2248×4000; la foto se reconoce por SHA-256\",\n    \"device\": \"GPU CUDA con bfloat16 nativo; se usa L4, como en las corridas anteriores (no se exige el nombre de la GPU)\"\n  },\n  \"call_plan\": [\n    {\n      \"call_id\": \"AMG-0\",\n      \"config_id\": \"AMG-0 BASE\",\n      \"generator_kwargs\": {\n        \"points_per_batch\": 64,\n        \"stability_score_offset\": 1.0,\n        \"mask_threshold\": 0.0,\n        \"box_nms_thresh\": 0.7,\n        \"crop_nms_thresh\": 0.7,\n        \"crop_overlap_ratio\": 0.3413333333333333,\n        \"point_grids\": null,\n        \"min_mask_region_area\": 0,\n        \"output_mode\": \"uncompressed_rle\",\n        \"use_m2m\": false,\n        \"multimask_output\": true,\n        \"points_per_side\": 32,\n        \"pred_iou_thresh\": 0.8,\n        \"stability_score_thresh\": 0.95,\n        \"crop_n_layers\": 0,\n        \"crop_n_points_downscale_factor\": 1\n      },\n      \"points_per_batch_fallback\": [\n        64,\n        32,\n        16\n      ]\n    },\n    {\n      \"call_id\": \"AMG-1\",\n      \"config_id\": \"AMG-1 DENSE\",\n      \"generator_kwargs\": {\n        \"points_per_batch\": 64,\n        \"stability_score_offset\": 1.0,\n        \"mask_threshold\": 0.0,\n        \"box_nms_thresh\": 0.7,\n        \"crop_nms_thresh\": 0.7,\n        \"crop_overlap_ratio\": 0.3413333333333333,\n        \"point_grids\": null,\n        \"min_mask_region_area\": 0,\n        \"output_mode\": \"uncompressed_rle\",\n        \"use_m2m\": false,\n        \"multimask_output\": true,\n        \"points_per_side\": 64,\n        \"pred_iou_thresh\": 0.8,\n        \"stability_score_thresh\": 0.95,\n        \"crop_n_layers\": 0,\n        \"crop_n_points_downscale_factor\": 1\n      },\n      \"points_per_batch_fallback\": [\n        64,\n        32,\n        16\n      ]\n    },\n    {\n      \"call_id\": \"AMG-2\",\n      \"config_id\": \"AMG-2 CROP\",\n      \"generator_kwargs\": {\n        \"points_per_batch\": 64,\n        \"stability_score_offset\": 1.0,\n        \"mask_threshold\": 0.0,\n        \"box_nms_thresh\": 0.7,\n        \"crop_nms_thresh\": 0.7,\n        \"crop_overlap_ratio\": 0.3413333333333333,\n        \"point_grids\": null,\n        \"min_mask_region_area\": 0,\n        \"output_mode\": \"uncompressed_rle\",\n        \"use_m2m\": false,\n        \"multimask_output\": true,\n        \"points_per_side\": 32,\n        \"pred_iou_thresh\": 0.8,\n        \"stability_score_thresh\": 0.95,\n        \"crop_n_layers\": 1,\n        \"crop_n_points_downscale_factor\": 2\n      },\n      \"points_per_batch_fallback\": [\n        64,\n        32,\n        16\n      ]\n    },\n    {\n      \"call_id\": \"AMG-3\",\n      \"config_id\": \"AMG-3 SENSITIVE\",\n      \"generator_kwargs\": {\n        \"points_per_batch\": 64,\n        \"stability_score_offset\": 1.0,\n        \"mask_threshold\": 0.0,\n        \"box_nms_thresh\": 0.7,\n        \"crop_nms_thresh\": 0.7,\n        \"crop_overlap_ratio\": 0.3413333333333333,\n        \"point_grids\": null,\n        \"min_mask_region_area\": 0,\n        \"output_mode\": \"uncompressed_rle\",\n        \"use_m2m\": false,\n        \"multimask_output\": true,\n        \"points_per_side\": 32,\n        \"pred_iou_thresh\": 0.7,\n        \"stability_score_thresh\": 0.9,\n        \"crop_n_layers\": 0,\n        \"crop_n_points_downscale_factor\": 1\n      },\n      \"points_per_batch_fallback\": [\n        64,\n        32,\n        16\n      ]\n    }\n  ],\n  \"operational\": {\n    \"points_per_batch\": \"empieza en 64; solo baja por OOM (64 → 32 → 16), y cada intento queda registrado (regla del sweep)\",\n    \"output\": \"uncompressed_rle por máscara, con su packed_sha256 (cabecera «HxW:» + np.packbits)\",\n    \"no_results_in_colab\": \"el cuaderno solo imprime el progreso; no muestra máscaras, número de máscaras ni scores\"\n  },\n  \"run_validity\": {\n    \"REAL_GPU_EVIDENCE\": \"integridad sin problemas, commit y checkpoint congelados, foto por hash, y device = cuda, dtype = bfloat16 y cuda_capability[0] ≥ 8\",\n    \"REAL_GPU_DIFFERENT_PRECISION_NOT_EVIDENCE\": \"CUDA sin bfloat16 (capacidad < 8, float16; p. ej. T4): no se lee\",\n    \"SIMULATED_RUN_NOT_EVIDENCE\": \"arnés con SAM simulado: prueba el cuaderno, nunca es resultado\",\n    \"REAL_CPU_NOT_EVIDENCE\": \"sin CUDA: no se lee\",\n    \"INVALID_BUNDLE\": \"manifiesto, kwargs, hashes de máscara o regla de points_per_batch rotos, o cuda_capability ausente o incoherente con el dtype (la regla del cuaderno: capacidad ≥ 8 → bfloat16, < 8 → float16): no se lee\"\n  },\n  \"rules\": {\n    \"R1\": {\n      \"name\": \"BOX_SCREEN_FAILURE\",\n      \"trigger\": \"objeto Tier A cuya mejor IoU de caja (caja declarada del inventario frente a mask_bbox de cada propuesta) es < 0.5 en las cuatro configuraciones\",\n      \"phases\": {\n        \"R1_TRIAGE\": {\n          \"candidates\": \"las 3 mejores propuestas por IoU de caja POR CONFIGURACIÓN (≤ 12), deduplicadas por packed_sha256; empates por índice menor\",\n          \"outcome\": \"si una llave dice COVERS_OBJECT o CANNOT_DETERMINE: NOT_CONFIRMED. Si las dos dicen MISS: ESCALATE_TO_EXHAUSTIVE, que todavía NO es un fallo\",\n          \"package\": \"PRAGMA_AE1S1_revision_ciega_R1.zip, una lámina Rnn.png por objeto; mapeo sellado sealed_mapping.json\"\n        },\n        \"R1_EXHAUSTIVE_BLIND_REVIEW\": {\n          \"when\": \"solo para los objetos que el triaje escala, y después de archivar las dos llaves del triaje\",\n          \"candidates\": \"TODAS las propuestas únicas de las cuatro configuraciones cuya mask_bbox tiene intersección de área no vacía con la caja congelada del objeto (cajas semiabiertas, la convención de box_iou); deduplicadas por packed_sha256; sin tope\",\n          \"pages\": \"láminas Rnn_pPP.png de hasta 12 candidatas, tantas como hagan falta; cada una repite el objeto\",\n          \"outcome\": \"CONFIRMED_BOX_SCREEN_FAILURE solo si las dos llaves revisan todas las láminas (reviewed_all_pages = true) y dicen MISS. Una candidata encontrada por cualquiera → NOT_CONFIRMED. CANNOT_DETERMINE, o un MISS sin reviewed_all_pages = true → NOT_CONFIRMED\",\n          \"package\": \"PRAGMA_AE1S1_revision_ciega_R1_exhaustiva.zip; mapeo sellado sealed_mapping_exhaustivo.json; mismas etiquetas de objeto Rnn que el triaje\"\n        }\n      },\n      \"blinding\": \"sin configuración, score SAM ni ranking; orden aleatorio con semilla fija (un generador nuevo para cada paquete); mapeo sellado fuera del paquete\",\n      \"sheet\": \"recorte del objeto con su caja en verde y cada candidata en magenta, con miniatura de la foto entera; etiquetas con nombre (ae0_044, ae0_045) tapadas siempre\",\n      \"seed\": 20260928,\n      \"answers\": [\n        \"COVERS_OBJECT\",\n        \"MISS\",\n        \"CANNOT_DETERMINE\"\n      ],\n      \"keys\": \"en cada fase, Claude compromete sus respuestas por SHA-256 antes de archivar las de ChatGPT; ChatGPT recibe solo el paquete ciego\",\n      \"wording\": \"BOX_SCREEN_FAILURE: el triaje no demuestra nada por sí solo; confirmado, significa que ninguna propuesta de este sweep contiene el objeto según las dos llaves\"\n    },\n    \"R2\": {\n      \"per_proposal\": \"iou_estimate, iou_min, iou_max: los de keydiff.iou_with_uncertainty (redondeo a 6 decimales), con U = uncertain_path de la persona y F = gt_mask ∖ U\",\n      \"per_person_config\": \"best_estimate = max iou_estimate (oráculo descriptivo); best_robust_min = max iou_min; best_possible_max = max iou_max; y los tres argmax (índice menor en empates)\",\n      \"threshold\": 0.7,\n      \"verdict\": {\n        \"PASS\": \"best_robust_min ≥ 0,70\",\n        \"FAIL\": \"best_possible_max < 0,70\",\n        \"DEPENDS_ON_UNCERTAINTY\": \"en otro caso\"\n      }\n    },\n    \"R3\": {\n      \"label\": \"FUSION_Q_ESTIMATE_BASED\",\n      \"params\": \"los del contrato: regla Q, τ = 0,10, erosión 5 px, banda de contacto 24 px, umbral 0,20\",\n      \"status\": \"diagnóstico: la fusión se mide contra la estimación binaria y no tiene cotas; nunca es early-stop por sí sola\"\n    },\n    \"R4\": {\n      \"FAIL_COMPONENT\": \"≥ 1 CONFIRMED_BOX_SCREEN_FAILURE, o ≥ 1 persona con R2 = FAIL en las cuatro configuraciones\",\n      \"otherwise\": \"INCONCLUSIVE_GT_INCOMPLETE → etapa 2 (máscaras de los 35 Tier A restantes, por doble llave)\",\n      \"never_fail\": \"un DEPENDS_ON_UNCERTAINTY nunca produce FAIL; un FAIL en menos de cuatro configuraciones tampoco\"\n    }\n  },\n  \"order\": [\n    \"1. integridad del ZIP sin mirar resultados (ae1_stage1.integrity)\",\n    \"2. disparadores de R1 (solo cajas); si los hay: triaje ciego → llave de Claude por hash → llave de ChatGPT\",\n    \"3. si el triaje escala algún objeto: revisión exhaustiva ciega → llave de Claude por hash → llave de ChatGPT\",\n    \"4. solo con R1 resuelto para todos los objetos disparados: R2, R3 y R4 (ae1_stage1.analyze)\",\n    \"5. informe con el techo de la etapa y la decisión de R4\"\n  ],\n  \"privacy\": \"las máscaras de SAM 2 derivan de la foto: van a local/, y a git solo hashes y cifras\",\n  \"content_sha256\": \"cee53ee86fa158dbf7398ae5c19d35cadc5ce5008920f671120502c5b45e271d\"\n}\n"
assert hashlib.sha256(PROTOCOL_TEXT.encode("utf-8")).hexdigest() == PROTOCOL_FILE_SHA256, "Protocolo alterado"
PROTOCOL = json.loads(PROTOCOL_TEXT)
assert PROTOCOL["status"] == "PREREGISTERED", "El protocolo no está prerregistrado"
CALL_PLAN = PROTOCOL["call_plan"]
assert [c["call_id"] for c in CALL_PLAN] == ["AMG-0", "AMG-1", "AMG-2", "AMG-3"]
FREEZE = PROTOCOL["freeze"]
assert FREEZE["SAM2_GIT_COMMIT"] == "2b90b9f5ceec907a1c18123530e92e794ad901a4"
assert FREEZE["CHECKPOINT_SHA256"] == "2647878d5dfa5098f2f8649825738a9345572bae2d4350a2468587ece47dd318"
assert FREEZE["IMAGE_SHA256"] == EXPECTED_IMAGE_SHA256
print("Protocolo verificado:", PROTOCOL["content_sha256"][:12], "…", "·", len(CALL_PLAN), "configuraciones")

In [ ]:
# Celda 4 · modelo sin postprocesado, como el ejemplo oficial de AMG en el commit congelado
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

if DEVICE != "cuda" or PRECISION["dtype"] != "bfloat16":
    raise RuntimeError("FAIL_ENVIRONMENT: hace falta CUDA en bfloat16 (capacidad ≥ 8, p. ej. L4); ver la celda 0")
t0 = time.perf_counter()
try:
    sam2_model = build_sam2(MODEL_CFG, str(CHECKPOINT), device=DEVICE, apply_postprocessing=False)
    synchronize()
except torch.cuda.OutOfMemoryError as exc:
    raise RuntimeError("FAIL_ENVIRONMENT: Large no cabe; usa L4 o A100. No se degrada a Small.") from exc
MODEL_LOAD_S = time.perf_counter() - t0
print(f"Modelo listo en {MODEL_LOAD_S:.1f} s")

In [ ]:
# Celda 5 · las 4 configuraciones del protocolo, en orden. Solo se imprime el progreso.
def rle_to_mask(rle):
    h, w = (int(v) for v in rle["size"])
    counts = np.asarray(rle["counts"], dtype=np.int64)
    values = np.zeros(counts.size, dtype=bool)
    values[1::2] = True
    flat = np.repeat(values, counts)
    assert flat.size == h * w, "RLE incoherente"
    return flat.reshape(w, h).T

def packed_sha256(mask):
    header = f"{mask.shape[0]}x{mask.shape[1]}:".encode()
    return hashlib.sha256(header + np.packbits(mask).tobytes()).hexdigest()

EXECUTED, RESULTS, MASK_HASHES = [], {}, {}
t_run = time.perf_counter()
for number, call in enumerate(CALL_PLAN, 1):
    attempts, anns = [], None
    for ppb in call["points_per_batch_fallback"]:      # solo baja por OOM (regla del sweep)
        attempts.append(ppb)
        generator = SAM2AutomaticMaskGenerator(sam2_model, **dict(call["generator_kwargs"], points_per_batch=ppb))
        try:
            synchronize(); start = time.perf_counter()
            with torch.inference_mode(), inference_precision():
                anns = generator.generate(image)
            synchronize(); seconds = time.perf_counter() - start
            break
        except torch.cuda.OutOfMemoryError:
            del generator
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            print(f"{call['call_id']}: sin memoria con points_per_batch={ppb}; se registra y se baja")
    if anns is None:
        raise RuntimeError(f"FAIL_ENVIRONMENT: {call['call_id']} no cabe ni con points_per_batch=16")
    masks = []
    for ann in anns:
        mask = rle_to_mask(ann["segmentation"])
        assert mask.shape == (2248, 4000), mask.shape
        masks.append({"rle": ann["segmentation"], "packed_sha256": packed_sha256(mask), "area": int(ann["area"]),
                      "bbox_xywh": [float(v) for v in ann["bbox"]], "predicted_iou": float(ann["predicted_iou"]),
                      "stability_score": float(ann["stability_score"]), "point_coords": ann["point_coords"],
                      "crop_box_xywh": [float(v) for v in ann["crop_box"]]})
    RESULTS[call["call_id"]] = masks
    MASK_HASHES[call["call_id"]] = [m["packed_sha256"] for m in masks]
    EXECUTED.append({"call_id": call["call_id"], "config_id": call["config_id"],
                     "generator_kwargs": dict(call["generator_kwargs"], points_per_batch=attempts[-1]),
                     "points_per_batch_attempts": attempts, "seconds": round(seconds, 3), "n_masks": len(masks)})
    print(f"configuración {number}/{len(CALL_PLAN)} hecha")
RUN_S = time.perf_counter() - t_run
print(f"Hecho en {RUN_S:.1f} s. No se muestran resultados: el análisis es externo y prerregistrado.")

In [ ]:
# Celda 6 · manifiesto, ZIP y descarga
def write_json(path, payload, compact=False):
    text = json.dumps(payload, separators=(",", ":")) if compact else json.dumps(payload, indent=2, ensure_ascii=False)
    path.write_text(text + "\n", encoding="utf-8")
    return path

(RUN_DIR / "masks").mkdir(exist_ok=True)
for call_id, masks in RESULTS.items():
    write_json(RUN_DIR / "masks" / f"{call_id}.json", {"call_id": call_id, "size": [2248, 4000], "masks": masks}, compact=True)
write_json(RUN_DIR / "ae1s1_config.json", {
    "notebook_version": "ae1s1-1.1", "run_id": RUN_ID, "protocol_file_sha256": PROTOCOL_FILE_SHA256,
    "protocol_content_sha256": PROTOCOL["content_sha256"], "environment": ENVIRONMENT,
    "gates": {"sam2_commit": "verificado en la celda 1", "checkpoint": "verificado en la celda 1",
              "image_sha256": "verificado en la celda 2", "protocol": "verificado en la celda 3"},
})
write_json(RUN_DIR / "ae1s1_calls.json", EXECUTED)
write_json(RUN_DIR / "ae1s1_report.json", {
    "run_id": RUN_ID, "status": "PENDING_ANALYSIS", "environment": ENVIRONMENT,
    "timings_s": {"model_load": round(MODEL_LOAD_S, 3), "run": round(RUN_S, 3)},
    "gpu_peak_gib": round(torch.cuda.max_memory_allocated() / 2**30, 3) if DEVICE == "cuda" else None,
    "note": "sin revisión en Colab: análisis externo según ae1/AE1_STAGE1_READING_PROTOCOL.json",
})
files_listed = sorted(p for p in RUN_DIR.rglob("*") if p.is_file() and p.suffix.lower() not in (".jpg", ".jpeg")
                      and p.name != "ae1s1_manifest.json")
manifest = {"run_id": RUN_ID, "files": {p.relative_to(RUN_DIR).as_posix(): {"bytes": p.stat().st_size, "sha256": sha256_file(p)}
                                        for p in files_listed},
            "masks": MASK_HASHES, "mask_hash": "sha256(«HxW:» + np.packbits(mask)), como pragma_ae.masks.packed_sha256"}
write_json(RUN_DIR / "ae1s1_manifest.json", manifest)
ZIP_PATH = WORK_DIR / f"PRAGMA_AE1S1_{RUN_ID}_PENDING_ANALYSIS.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for p in files_listed + [RUN_DIR / "ae1s1_manifest.json"]:
        archive.write(p, p.relative_to(RUN_DIR).as_posix())
with zipfile.ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None
ZIP_SHA256 = sha256_file(ZIP_PATH)
print(f"ZIP: {ZIP_PATH.name} · {ZIP_PATH.stat().st_size / 2**20:.1f} MiB · SHA-256 {ZIP_SHA256}")
if not PRAGMA_HEADLESS:
    files.download(str(ZIP_PATH))
print("Adjunta este ZIP solo a Claude. No compartas capturas de esta corrida.")

## 7. Qué hacer ahora

1. Busca el ZIP `PRAGMA_AE1S1_…_PENDING_ANALYSIS.zip` en tu carpeta de descargas.
2. **Adjúntalo a Claude** en la sesión de Claude Code. No se lo mandes a ChatGPT.
3. Claude comprueba la integridad sin mirar resultados. Si alguna caja dispara el cribado, prepara
   el paquete ciego: es lo único que recibe ChatGPT, sin carta.